## Modeling

In [1]:
DATA_FILE = "Data/all_dialogs.txt"

all_dialogs = open(DATA_FILE)

text = all_dialogs.read()

dialogs = text.split("-" * 100)

print(len(dialogs))

3236


In [2]:
#print(dialogs[9])

## Implementation

In [3]:
dialog_state = {
    "state": 1,
    "area": None,
    "food": None,
    "pricerange": None,
    "restaurant": None,
    "alternatives": []
}

### Slot extraction

In [4]:
import pandas as pd

RESTAURANT_FILE = "Data/restaurant_info.csv"

restaurants = pd.read_csv(RESTAURANT_FILE)

#print(restaurants.columns)
#restaurants.head()

In [5]:
areas = restaurants["area"].dropna().unique().tolist()
foods = restaurants["food"].dropna().unique().tolist()
price_ranges = restaurants["pricerange"].dropna().unique().tolist()

#print(areas)
#print(price_ranges)
#print(foods)


In [6]:
### 1. Keyword matching

def keyword_slot_extractor(utterance):
    utterance = utterance.lower()
    
    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    for area in areas:
        if area in utterance:
            slots["area"] = area

    for food in foods:
        if food in utterance:
            slots["food"] = food

    for price in price_ranges:
        if price in utterance:
            slots["pricerange"] = price
            
    
    return slots

In [7]:
### 2. Levenshtein edit distance

#%pip install levenshtein
from Levenshtein import distance

def levenshtein_slot_extractor(utterance):
    utterance = utterance.lower()
    words = utterance.split()

    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    # Area
    best_match = None
    best_distance = float("inf")

    for area in areas:
        for word in words:
            current_distance = distance(word, area)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = area

    slots["area"] = best_match        
    
    
    # Food
    best_match = None
    best_distance = float("inf")

    for food in foods:
        for word in words:
            current_distance = distance(word, food)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = food

    slots["food"] = best_match
    

    # Pricerange
    best_match = None
    best_distance = float("inf")

    for price in price_ranges:
        for word in words:
            current_distance = distance(word, price)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = price

    slots["pricerange"] = best_match

    return slots

In [8]:
### 3. Semantic similarity via embeddings

## TODO

### Response generation

Use string templates for all system utterances, either simple strings or patterns with variables to fill in. For example:

"askpricerange": "Which price range do you prefer?"
"confirmfoodtype": "I did not recognize {givenfoodtype}. Did you mean {correctedfoodtype}?"

In [9]:
## TODO

### Restaurant lookup

Once all preferences are collected, the system provides a recommendation. Write a Python function that retrieves matching restaurants from restaurant_info.csv according to the extracted preferences. If no restaurant matches, the system should inform the user. If multiple match, one is chosen at random and the others are stored so the user can ask for an alternative using the same preferences.

In [12]:
import random
import pandas as pd

restaurants = pd.read_csv("Data/restaurant_info.csv")

def restaurant_lookup(area, food, pricerange):

    matches = restaurants[(restaurants["area"] == area) & (restaurants["food"] == food) & (restaurants["pricerange"] == pricerange)]
    matches = matches["restaurantname"].tolist()

    if len(matches) == 0:
        return "No matches", []
    
    
    recommendation = random.choice(matches)
    
    alternatives = [restaurant for restaurant in matches if restaurant != recommendation]
        
    return recommendation, alternatives


#recommendation, alternatives = restaurant_lookup("north", "indian", "moderate")
#print(recommendation)
#print(alternatives)

### Reasoning component